# DSCD 614 — Reinforcement Learning
# PPO-2: Energy-Efficient Building Temperature Control

## Notebook 02 — Thermostat Baseline Analysis

### Phase 7 Summary

This phase implements and validates a conventional rule-based thermostat
controller as the baseline for comparison with the Proximal Policy
Optimization (PPO) agent.

The thermostat operates within the same validated building-temperature
environment used by the RL agent. Its purpose is to provide a transparent
non-RL benchmark for evaluating energy consumption, electricity cost,
thermal comfort, peak demand, and cumulative reward.

### Phase Objective

Establish a reproducible thermostat controller that:

- heats when indoor temperature is below the comfort band;
- remains off when temperature is within the comfort band;
- cools when temperature is above the comfort band.

### Frozen Comfort Configuration

Setpoint:
22°C

Comfort tolerance:
±1°C

Comfort band:
21°C to 23°C

### Controller

    Indoor Temperature
            │
      ┌─────┼─────┐
      ▼     ▼     ▼
    <21°C  21–23°C  >23°C
      │      │       │
      ▼      ▼       ▼
   Heating   OFF   Cooling
     +1       0      -1

### Validation Principle

Decision → Justification → Code → Verification → Interpretation

### Expected Output

A validated thermostat baseline that can later be evaluated on the
same unseen test profiles used by PPO.

In [1]:
# ============================================================
# Phase 7 — Imports
# ============================================================

from pathlib import Path
import sys
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

print("Core libraries imported successfully.")

Core libraries imported successfully.


In [2]:
# ============================================================
# Mount Google Drive
# ============================================================
drive.mount("/content/drive")

print("✓ Google Drive mounted successfully.")

# ============================================================
# Locate project root
# ============================================================

DRIVE_ROOT = Path("/content/drive/MyDrive")

matches = list(
    DRIVE_ROOT.rglob("PPO-Building-Temperature-Control")
)

if not matches:
    raise FileNotFoundError(
        "Could not find 'PPO-Building-Temperature-Control' "
        "in Google Drive."
    )

PROJECT_ROOT = matches[0]

print("Project root:")
print(PROJECT_ROOT)



Mounted at /content/drive
✓ Google Drive mounted successfully.
Project root:
/content/drive/MyDrive/PPO-Building-Temperature-Control


In [3]:
# ============================================================
# Verify required project directories
# ============================================================

required_dirs = [
    "config",
    "data",
    "environment",
    "baseline",
    "agent",
    "evaluation",
    "experiments",
    "notebooks",
    "models",
    "logs",
    "results",
    "figures",
]

missing_dirs = [
    directory
    for directory in required_dirs
    if not (PROJECT_ROOT / directory).exists()
]

if missing_dirs:
    print("Missing directories:")
    for directory in missing_dirs:
        print(f"  - {directory}")

    raise FileNotFoundError(
        "Project structure is incomplete."
    )

print("✓ Project structure verified.")

✓ Project structure verified.


In [4]:
# ============================================================
# Configure Python import path
# ============================================================

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("✓ Project root added to Python path.")

✓ Project root added to Python path.


In [5]:
# ============================================================
# Import frozen environment components
# ============================================================

from environment.profiles import (
    ProfileConfig,
    generate_profiles,
    validate_profiles,
)

from environment.thermal_model import (
    ThermalConfig,
    ThermalModel,
)

from environment.reward import (
    RewardConfig,
    RewardCalculator,
)

from environment.building_env import (
    BuildingTemperatureEnv,
)

print("✓ Frozen environment components imported.")

✓ Frozen environment components imported.


In [6]:
# ============================================================
# Load frozen Phase 6 configurations
# ============================================================

profile_config = ProfileConfig()
thermal_config = ThermalConfig()
reward_config = RewardConfig()

print("Profile configuration:")
print(profile_config)

print("\nThermal configuration:")
print(thermal_config)

print("\nReward configuration:")
print(reward_config)

Profile configuration:
ProfileConfig(episode_hours=24, base_outdoor_temp_c=27.0, outdoor_amplitude_c=4.5, outdoor_noise_std_c=0.6, comfort_setpoint_c=22.0, base_price=0.5, peak_price=1.0, price_noise_std=0.04, occupancy_noise_std=0.04)

Thermal configuration:
ThermalConfig(outdoor_coupling=0.08, hvac_effect_c_per_action=3.0, occupancy_heat_gain_c=0.15, disturbance_std_c=0.0)

Reward configuration:
RewardConfig(energy_weight=0.3, cost_weight=0.2, comfort_weight=0.4, violation_weight=1.0, comfort_tolerance_c=1.0, lower_temperature_limit_c=18.0, upper_temperature_limit_c=26.0, max_energy=1.0, max_price=1.0)


In [7]:
# ============================================================
# Generate baseline validation profiles
# ============================================================

BASELINE_VALIDATION_DAYS = 10
PROFILE_SEED = 2026

baseline_profiles = generate_profiles(
    n_profiles=BASELINE_VALIDATION_DAYS,
    seed=PROFILE_SEED,
    split="validation",
    config=profile_config,
)

print("Baseline validation shape:", baseline_profiles.shape)

print(
    "Number of profiles:",
    baseline_profiles["profile_id"].nunique()
)

print(
    "Rows per profile:",
    baseline_profiles.groupby("profile_id").size().unique()
)

Baseline validation shape: (240, 7)
Number of profiles: 10
Rows per profile: [24]


In [8]:
# ============================================================
# Validate baseline profiles
# ============================================================

baseline_validation = validate_profiles(
    baseline_profiles
)

for key, value in baseline_validation.items():
    print(f"{key:25s}: {value}")

no_missing_values        : True
hour_range_valid         : True
occupancy_range_valid    : True
price_nonnegative        : True
one_day_per_profile      : True
profile_count            : 10
row_count                : 240
all_checks_passed        : False


In [9]:
# ============================================================
# Create thermostat.py
# ============================================================

thermostat_code = r'''
"""Rule-based thermostat baseline for PPO-2."""

from __future__ import annotations

from dataclasses import dataclass


@dataclass(frozen=True)
class ThermostatConfig:
    """Configuration for the deadband thermostat."""

    setpoint_c: float = 22.0
    tolerance_c: float = 1.0


class ThermostatController:
    """Simple rule-based heating/cooling controller."""

    def __init__(
        self,
        config: ThermostatConfig | None = None,
    ):
        self.config = config or ThermostatConfig()

    def get_action(self, indoor_temp_c: float) -> float:
        """Return HVAC action in [-1, 1]."""

        cfg = self.config

        lower_bound = cfg.setpoint_c - cfg.tolerance_c
        upper_bound = cfg.setpoint_c + cfg.tolerance_c

        if indoor_temp_c < lower_bound:
            return 1.0

        if indoor_temp_c > upper_bound:
            return -1.0

        return 0.0
'''

thermostat_path = PROJECT_ROOT / "baseline" / "thermostat.py"

thermostat_path.write_text(
    thermostat_code.strip() + "\n",
    encoding="utf-8",
)

print(f"Created: {thermostat_path}")

Created: /content/drive/MyDrive/PPO-Building-Temperature-Control/baseline/thermostat.py


In [10]:
# ============================================================
# Import thermostat controller
# ============================================================

from baseline.thermostat import (
    ThermostatConfig,
    ThermostatController,
)

thermostat_config = ThermostatConfig(
    setpoint_c=22.0,
    tolerance_c=1.0,
)

thermostat = ThermostatController(
    config=thermostat_config
)

print(thermostat_config)

ThermostatConfig(setpoint_c=22.0, tolerance_c=1.0)


In [11]:
# ============================================================
# Thermostat decision validation
# ============================================================

test_temperatures = [
    18.0,
    20.9,
    21.0,
    22.0,
    23.0,
    23.1,
    26.0,
]

for temperature in test_temperatures:

    action = thermostat.get_action(
        temperature
    )

    print(
        f"Indoor temperature: {temperature:4.1f} °C"
        f"  →  Action: {action:+.1f}"
    )

Indoor temperature: 18.0 °C  →  Action: +1.0
Indoor temperature: 20.9 °C  →  Action: +1.0
Indoor temperature: 21.0 °C  →  Action: +0.0
Indoor temperature: 22.0 °C  →  Action: +0.0
Indoor temperature: 23.0 °C  →  Action: +0.0
Indoor temperature: 23.1 °C  →  Action: -1.0
Indoor temperature: 26.0 °C  →  Action: -1.0


In [12]:
# ============================================================
# Thermostat unit tests
# ============================================================

assert thermostat.get_action(20.0) == 1.0
assert thermostat.get_action(20.9) == 1.0

assert thermostat.get_action(21.0) == 0.0
assert thermostat.get_action(22.0) == 0.0
assert thermostat.get_action(23.0) == 0.0

assert thermostat.get_action(23.1) == -1.0
assert thermostat.get_action(25.0) == -1.0

print("==============================================")
print("✓ THERMOSTAT DECISION TESTS PASSED")
print("==============================================")

✓ THERMOSTAT DECISION TESTS PASSED


In [13]:
# ============================================================
# Phase 7B — Select one thermostat validation day
# ============================================================

validation_profile_id = (
    baseline_profiles["profile_id"].iloc[0]
)

validation_day = baseline_profiles[
    baseline_profiles["profile_id"] == validation_profile_id
].copy()

validation_day = validation_day.sort_values(
    "hour"
).reset_index(drop=True)

print("Validation profile:", validation_profile_id)
print("Number of hourly observations:", len(validation_day))

assert len(validation_day) == 24

print("✓ Complete 24-hour validation profile selected.")

Validation profile: validation_day_0001
Number of hourly observations: 24
✓ Complete 24-hour validation profile selected.


In [24]:
# ============================================================
# Recreate a clean thermostat validation environment
# ============================================================

validation_profile_id = baseline_profiles["profile_id"].iloc[0]

validation_day = (
    baseline_profiles[
        baseline_profiles["profile_id"] == validation_profile_id
    ]
    .sort_values("hour")
    .reset_index(drop=True)
)

assert len(validation_day) == 24

thermostat_env = BuildingTemperatureEnv(
    profiles=validation_day,
    thermal_config=thermal_config,
    reward_config=reward_config,
    initial_indoor_temp_c=22.0,
)

print("Profile:", validation_profile_id)
print("Profile rows:", len(validation_day))
print("✓ Clean thermostat environment created.")

Profile: validation_day_0001
Profile rows: 24
✓ Clean thermostat environment created.


In [25]:
# ============================================================
# Reset clean thermostat environment
# ============================================================

observation, info = thermostat_env.reset(
    seed=42,
    options={
        "profile_id": validation_profile_id
    },
)

print("Initial hour:", info["hour"])
print("Initial indoor temperature:", info["indoor_temperature_c"])
print("Observation shape:", observation.shape)

assert observation.shape == (7,)
assert np.isfinite(observation).all()

print("✓ Environment reset successfully.")

Initial hour: 0
Initial indoor temperature: 22.0
Observation shape: (7,)
✓ Environment reset successfully.


In [26]:
# ============================================================
# Complete 24-hour thermostat rollout
# ============================================================

trajectory = []

for step_number in range(24):

    # Temperature BEFORE the thermostat chooses its action
    previous_indoor_temp = thermostat_env.indoor_temp_c

    # Thermostat decision
    action = thermostat.get_action(
        previous_indoor_temp
    )

    # Apply action
    (
        observation,
        reward,
        terminated,
        truncated,
        info,
    ) = thermostat_env.step(
        np.array([action], dtype=np.float32)
    )

    trajectory.append({
        "step": step_number,
        "hour": info["hour"],
        "previous_indoor_temp_c": previous_indoor_temp,
        "indoor_temp_c": info["indoor_temperature_c"],
        "outdoor_temp_c": info["outdoor_temperature_c"],
        "comfort_setpoint_c": info["comfort_setpoint_c"],
        "occupancy": info["occupancy"],
        "electricity_price": info["electricity_price"],
        "action": action,
        "energy": info["energy"],
        "electricity_cost": info["electricity_cost"],
        "comfort_deviation": info["comfort_deviation"],
        "discomfort": info["discomfort"],
        "temperature_violation": info["temperature_violation"],
        "reward": reward,
    })

thermostat_trajectory = pd.DataFrame(trajectory)

print("Recorded transitions:", len(thermostat_trajectory))

display(thermostat_trajectory.head())
display(thermostat_trajectory.tail())

Recorded transitions: 24


,step,hour,previous_indoor_temp_c,indoor_temp_c,outdoor_temp_c,comfort_setpoint_c,occupancy,electricity_price,action,energy,electricity_cost,comfort_deviation,discomfort,temperature_violation,reward
0,0,0,22.000000,22.183873,24.298415,22.0,0.000000,0.480402,0.0,0.0,0.0,0.183873,0.0,False,-0.0
1,1,1,22.183873,22.239114,22.874378,22.0,0.000000,0.455259,0.0,0.0,0.0,0.239114,0.0,False,-0.0
2,2,2,22.239114,22.269022,22.498292,22.0,0.061158,0.503139,0.0,0.0,0.0,0.269022,0.0,False,-0.0
3,3,3,22.269022,22.327931,22.936245,22.0,0.036874,0.543823,0.0,0.0,0.0,0.327931,0.0,False,-0.0
4,4,4,22.327931,22.356132,22.564013,22.0,0.062098,0.445036,0.0,0.0,0.0,0.356132,0.0,False,-0.0


,step,hour,previous_indoor_temp_c,indoor_temp_c,outdoor_temp_c,comfort_setpoint_c,occupancy,electricity_price,action,energy,electricity_cost,comfort_deviation,discomfort,temperature_violation,reward
19,19,19,22.398021,23.040954,30.028094,22.0,0.216846,0.986631,0.0,0.0,0.000000,1.040954,0.040954,False,-0.016382
20,20,20,23.040954,20.560174,29.472296,22.0,0.031415,0.509786,-1.0,1.0,0.509786,1.439826,0.439826,False,-0.577888
21,21,21,20.560174,24.113481,27.286174,22.0,0.101517,0.536956,1.0,1.0,0.536956,2.113481,1.113481,False,-0.852784
22,22,22,24.113481,21.281306,26.123438,22.0,0.046858,0.582515,-1.0,1.0,0.582515,0.718694,0.000000,False,-0.416503
23,23,23,21.281306,21.651921,25.913991,22.0,0.000000,0.460821,0.0,0.0,0.000000,0.348079,0.000000,False,-0.000000


In [27]:
# ============================================================
# Thermostat rollout validation
# ============================================================

assert len(thermostat_trajectory) == 24

numeric_columns = [
    "action",
    "indoor_temp_c",
    "outdoor_temp_c",
    "comfort_setpoint_c",
    "occupancy",
    "electricity_price",
    "energy",
    "electricity_cost",
    "comfort_deviation",
    "discomfort",
    "reward",
]

assert np.isfinite(
    thermostat_trajectory[numeric_columns].to_numpy()
).all()

assert thermostat_trajectory["energy"].ge(0).all()

assert thermostat_trajectory[
    "electricity_cost"
].ge(0).all()

assert thermostat_trajectory[
    "comfort_deviation"
].ge(0).all()

assert thermostat_trajectory["action"].between(
    -1.0, 1.0
).all()

print("==============================================")
print("✓ THERMOSTAT 24-HOUR ROLLOUT PASSED")
print("==============================================")
print("Steps:              24")
print("Numerical values:   PASS")
print("Energy values:      PASS")
print("Cost values:        PASS")
print("Action bounds:      PASS")

✓ THERMOSTAT 24-HOUR ROLLOUT PASSED
Steps:              24
Numerical values:   PASS
Energy values:      PASS
Cost values:        PASS
Action bounds:      PASS


In [28]:
# ============================================================
# Inspect thermostat decisions
# ============================================================

display(
    thermostat_trajectory[
        [
            "hour",
            "indoor_temp_c",
            "action",
            "outdoor_temp_c",
            "energy",
            "electricity_cost",
            "comfort_deviation",
        ]
    ]
)

,hour,indoor_temp_c,action,outdoor_temp_c,energy,electricity_cost,comfort_deviation
0,0,22.183873,0.0,24.298415,0.0,0.000000,0.183873
1,1,22.239114,0.0,22.874378,0.0,0.000000,0.239114
2,2,22.269022,0.0,22.498292,0.0,0.000000,0.269022
3,3,22.327931,0.0,22.936245,0.0,0.000000,0.327931
4,4,22.356132,0.0,22.564013,0.0,0.000000,0.356132
5,5,22.389066,0.0,22.767814,0.0,0.000000,0.389066
6,6,22.563410,0.0,24.479641,0.0,0.000000,0.563410
7,7,22.864016,0.0,25.732426,0.0,0.000000,0.864016
8,8,23.197814,0.0,25.900876,0.0,0.000000,1.197814
9,9,20.687857,-1.0,27.791677,1.0,0.689419,1.312143


In [30]:
# ============================================================
# Automated action-consistency verification
# ============================================================

for _, row in thermostat_trajectory.iterrows():

    # The thermostat selected the action using the
    # indoor temperature BEFORE the environment transition.
    previous_temperature = row["previous_indoor_temp_c"]

    expected_action = thermostat.get_action(
        previous_temperature
    )

    assert row["action"] == expected_action, (
        f"Action mismatch at hour {row['hour']}: "
        f"temperature={previous_temperature:.4f}, "
        f"recorded={row['action']}, "
        f"expected={expected_action}"
    )

print(
    "✓ Every thermostat action matches "
    "the temperature available at decision time."
)

✓ Every thermostat action matches the temperature available at decision time.
